In [ ]:
!pip install transformers torch accelerate


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
import numpy as np


In [ ]:
df = pd.read_csv("/content/match2coach_realistic_100k.csv")
df.head()


,text,event_vec,event_type,league,correctness,tactical,emotion
0,P. Müller plays a simple pass into midfield.,"[0, 1, 0, 0.251]",pass,UEFA_CL,1,0,3.18
1,L. Rossi lashes one toward goal and it's put w...,"[0, 0, 0, 0.592]",shot,EPL,1,1,2.44
2,"Dropping his shoulder, A. Silva bends the ball...","[0, 1, 0, 0.678]",cross,Ligue1,1,0,2.43
3,R. Gomes meets the ball with his head but it g...,"[0, 1, 0, 0.373]",header,International,1,1,3.61
4,A remarkable reflex save that keeps the scores...,"[1, 0, 0, 0.547]",save,UEFA_CL,1,1,4.49


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

class TextOnlyDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text_inputs = self.tokenizer(
            row["text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        event_vec = torch.tensor(eval(row["event_vec"]), dtype=torch.float)

        labels = {
            "correctness": torch.tensor(row["correctness"], dtype=torch.long),
            "tactical": torch.tensor(row["tactical"], dtype=torch.long),
            "emotion": torch.tensor(row["emotion"], dtype=torch.float)
        }

        return text_inputs, event_vec, labels


In [ ]:
dataset = TextOnlyDataset(df, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
len(dataset)


100000

In [ ]:
class TextOnlyMatch2Coach(nn.Module):
    def __init__(self, text_dim=768, event_dim=4, hidden_dim=512):
        super().__init__()

        self.text_encoder = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.event_proj = nn.Linear(event_dim, hidden_dim)
        self.text_proj = nn.Linear(text_dim, hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=8,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

        self.correctness_head = nn.Linear(hidden_dim, 3)
        self.tactical_head = nn.Linear(hidden_dim, 3)
        self.emotion_head = nn.Linear(hidden_dim, 1)

    def forward(self, text_inputs, event_vec):
        txt_emb = self.text_encoder(
            input_ids=text_inputs["input_ids"].squeeze(1),
            attention_mask=text_inputs["attention_mask"].squeeze(1)
        ).last_hidden_state[:,0,:]

        txt_emb = self.text_proj(txt_emb)
        evt_emb = self.event_proj(event_vec)

        fusion = torch.stack([txt_emb, evt_emb], dim=1)
        fused = self.transformer(fusion).mean(dim=1)

        return (
            self.correctness_head(fused),
            self.tactical_head(fused),
            self.emotion_head(fused)
        )


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TextOnlyMatch2Coach().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
ce = nn.CrossEntropyLoss()
mse = nn.MSELoss()


In [ ]:
epochs = 4

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in dataloader:
        text_inputs, event_vec, labels = batch

        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
        event_vec = event_vec.to(device)

        optimizer.zero_grad()

        c_pred, t_pred, e_pred = model(text_inputs, event_vec)

        loss_c = ce(c_pred, labels["correctness"].to(device))
        loss_t = ce(t_pred, labels["tactical"].to(device))
        loss_e = mse(e_pred.squeeze(), labels["emotion"].to(device))

        loss = 0.4*loss_c + 0.35*loss_t + 0.25*loss_e

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(dataloader):.4f}")


Epoch 1/4 — Loss: 0.3722
Epoch 2/4 — Loss: 0.3637
Epoch 3/4 — Loss: 0.3621
Epoch 4/4 — Loss: 0.3608


In [ ]:
model.eval()

sample = dataset[3]
text_inputs, event_vec, labels = sample
text_inputs = {k:v.to(device) for k,v in text_inputs.items()}
event_vec = event_vec.unsqueeze(0).to(device)

c, t, e = model(text_inputs, event_vec)

print("Prediction:")
print("Correctness:", c.argmax().item(), "True:", labels["correctness"].item())
print("Tactical:",   t.argmax().item(), "True:", labels["tactical"].item())
print("Emotion:",    float(e.item()), "True:", labels["emotion"].item())


Prediction:
Correctness: 1 True: 1
Tactical: 1 True: 1
Emotion: 3.00034236907959 True: 3.609999895095825
